In [93]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/1976-2024-house.tab")

# some cols both True and "True" values
bool_cols = ['runoff', 'special', 'writein', 'unofficial', 'fusion_ticket']
for col in bool_cols:
    df[col] = df[col].astype(str).str.strip().str.upper() == 'TRUE'

print(df.shape)
print(df.head(2))
# print(df.dtypes)

(33805, 20)
   year    state state_po  state_fips  state_cen  state_ic    office  \
0  1976  ALABAMA       AL           1         63        41  US HOUSE   
1  1976  ALABAMA       AL           1         63        41  US HOUSE   

   district stage  runoff  special       candidate       party  writein  \
0         1   GEN   False    False  BILL DAVENPORT    DEMOCRAT    False   
1         1   GEN   False    False    JACK EDWARDS  REPUBLICAN    False   

    mode  candidatevotes  totalvotes  unofficial   version  fusion_ticket  
0  TOTAL           58906      157170       False  20250910          False  
1  TOTAL           98257      157170       False  20250910          False  


C:\Users\Таня\AppData\Local\Temp\ipykernel_16856\3687504984.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/1976-2024-house.tab")


In [94]:
# Alabama is roughly 7 times bigger than Alaska population-wise and that's why there are 7x districts
print(df[(df['state_po'] == 'AL') & (df['year'] == 2024) & (df['stage'] == 'GEN')]['district'].unique())
print(df[(df['state_po'] == 'AK') & (df['year'] == 2024) & (df['stage'] == 'GEN')]['district'].unique())

[1 2 3 4 5 6 7]
[0]


### Drop useless columns ###

In [95]:
for col in df.columns:
    print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts().head(10))

# first three are just a different way to encode state_po
# fusion_ticket is a labeling quirk (mainly NY), doesn't change who won
useless_cols = ["state_fips", "state_cen", "state_ic", "fusion_ticket"]

# these cols have just one value - drop them
cols_with_one_value = ["office", "mode", "version"]

# these cols have two values, filter out the minor category, then drop the col
# stage == PRI → intra-party contests (e.g. two Democrats fighting with each other)
# special == TRUE → special elections (vacancy mid-term), different dynamics than regular elections
# runoff == TRUE → a second round after no one hit a threshold, mostly a Southern thing
# unofficial == TRUE → uncertified results
# writein == TRUE → filter out, write-in candidates never win and distort vote totals
filter_then_drop = ["stage", "special", "runoff", "unofficial", "writein"]


--- year (25 unique values) ---
1996    1660
2000    1608
1992    1524
2010    1500
2020    1467
2016    1440
2002    1433
2012    1427
2008    1381
2006    1374
Name: year, dtype: int64

--- state (51 unique values) ---
NEW YORK        4137
CALIFORNIA      3447
TEXAS           2089
MICHIGAN        1641
NEW JERSEY      1332
FLORIDA         1290
ILLINOIS        1261
PENNSYLVANIA    1232
OHIO            1192
VIRGINIA         878
Name: state, dtype: int64

--- state_po (51 unique values) ---
NY    4137
CA    3447
TX    2089
MI    1641
NJ    1332
FL    1290
IL    1261
PA    1232
OH    1192
VA     878
Name: state_po, dtype: int64

--- state_fips (51 unique values) ---
36    4137
6     3447
48    2089
26    1641
34    1332
12    1290
17    1261
42    1232
39    1192
51     878
Name: state_fips, dtype: int64

--- state_cen (51 unique values) ---
21    4137
93    3447
74    2089
34    1641
22    1332
59    1290
33    1261
23    1232
31    1192
54     878
Name: state_cen, dtype: int64

--- sta

In [96]:
if "stage" in df.columns:
    df = df[df["stage"] == "GEN"].copy()

for col in filter_then_drop[1:]:
    if col in df.columns:
        df = df[df[col] == False]

cols_to_drop = [c for c in useless_cols + cols_with_one_value + filter_then_drop if c in df.columns]
df = df.drop(columns=cols_to_drop).reset_index(drop=True).copy()

print(df.shape)
print(df.columns.tolist())

(30824, 8)
['year', 'state', 'state_po', 'district', 'candidate', 'party', 'candidatevotes', 'totalvotes']


### Filter out NA values ###

In [97]:
print(df.isnull().sum())
df[df['party'].isnull()]['candidate'].value_counts()

year                 0
state                0
state_po             0
district             0
candidate            0
party             1290
candidatevotes       0
totalvotes           0
dtype: int64


OTHER                              420
BLANK VOTE/SCATTERING              373
SCATTERING                          68
BLANK VOTE                          65
VOID                                55
BLANK                               54
UNDERVOTES                          54
BLANK VOTE/VOID VOTE/SCATTERING     54
BLANKS                              18
ALL OTHERS                          18
OVERVOTES                           18
OVER VOTE                           16
VOID VOTE                           16
MISCELLANEOUS                       12
OVER VOTES                           9
OTHER WRITE-INS                      8
BLANK VOTES                          6
UNDER VOTES                          6
WORKING FAMILIES                     4
INDEPENDENT                          3
SCATTER                              2
ERNEST JOHNSON                       1
RANDY ALTSCHULER                     1
DAN HILL                             1
MARTIN JAMES MONROE                  1
MODERATE                 

In [98]:
# Even the real ones (INDEPENDENT, WORKING FAMILIES) almost never win House seats,
# so they won't affect a D vs R prediction
df = df[df['party'].notna()].reset_index(drop=True)
print(df.shape)
print(df.dtypes)

(29534, 8)
year               int64
state             object
state_po          object
district           int64
candidate         object
party             object
candidatevotes     int64
totalvotes         int64
dtype: object


In [99]:
df.describe()

,year,district,candidatevotes,totalvotes
count,29534.000000,29534.000000,29534.000000,29534.000000
mean,2000.363513,10.075201,77488.540868,218320.248324
std,14.172550,10.212681,66151.552844,79923.592746
min,1976.000000,0.000000,-1.000000,-1.000000
25%,1988.000000,3.000000,8678.250000,163389.500000
50%,2000.000000,6.000000,72515.500000,208776.000000
75%,2012.000000,14.000000,121843.250000,268524.000000
max,2024.000000,53.000000,387109.000000,656104.000000


In [100]:
df = df[(df['candidatevotes'] >= 0) & (df['totalvotes'] >= 0)].reset_index(drop=True)
print(df.shape)
print(sorted(df['year'].unique()))

party_map = {
    'DEMOCRATIC-FARMER-LABOR': 'DEMOCRAT',
    'FOGLIETTA (DEMOCRAT)': 'DEMOCRAT',
    'INDEPENDENT-REPUBLICAN': 'REPUBLICAN',
}
df['party'] = df['party'].replace(party_map)

# keep only the top candidate per party per race — correct handling of rare multi-candidate cases
party_votes = df.loc[
    df.groupby(['year', 'state_po', 'district', 'party'])['candidatevotes'].idxmax()
][['year', 'state_po', 'district', 'party', 'candidatevotes', 'totalvotes']].reset_index(drop=True)

dems = party_votes[party_votes['party'] == 'DEMOCRAT'].copy()
dems['dem_vote_share'] = dems['candidatevotes'] / dems['totalvotes']

dems = dems.sort_values(['state_po', 'district', 'year'])
# 0.5 -> no lean, <0.3 heavy rep_lean, >0.7 heavy dem_lean
dems['dem_lean'] = (
    dems.groupby(['state_po', 'district'])['dem_vote_share']
    .transform(lambda x: x.shift(1).rolling(2).mean())
)
print(dems.shape)

(29531, 8)
[1976, 1978, 1980, 1982, 1984, 1986, 1988, 1990, 1992, 1994, 1996, 1998, 2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]
(10281, 8)


### Find winners ###

In [101]:
winners = party_votes.loc[party_votes.groupby(['year', 'state_po', 'district'])['candidatevotes'].idxmax()].reset_index(drop=True)
winners['vote_share'] = winners['candidatevotes'] / winners['totalvotes']
winners = winners.merge(
    dems[['year', 'state_po', 'district', 'dem_lean']],
    on=['year', 'state_po', 'district'],
    how='left'
)

print(winners.shape)
print(winners['dem_lean'].isna().sum())
print(winners.head())

(10841, 8)
1572
   year state_po  district       party  candidatevotes  totalvotes  \
0  1976       AK         0  REPUBLICAN           83722      118208   
1  1976       AL         1  REPUBLICAN           98257      157170   
2  1976       AL         2  REPUBLICAN           90069      156362   
3  1976       AL         3    DEMOCRAT          106935      108048   
4  1976       AL         4    DEMOCRAT          141490      176022   

   vote_share  dem_lean  
0    0.708260       NaN  
1    0.625164       NaN  
2    0.576029       NaN  
3    0.989699       NaN  
4    0.803820       NaN  


### Feature generation ###

In [102]:
# MIT data has ~3% gaps for years 1996 and 2018
seats_per_year = winners.groupby(['year', 'party'])['district'].count().unstack(fill_value=0)
seats_per_year = seats_per_year[['DEMOCRAT', 'REPUBLICAN']]
seats_per_year["Total"] = seats_per_year["DEMOCRAT"] + seats_per_year["REPUBLICAN"]
seats_per_year["Majority"] = np.where(seats_per_year["DEMOCRAT"] > seats_per_year["REPUBLICAN"], "DEMOCRAT", "REPUBLICAN")

print(len(seats_per_year))
print(seats_per_year)

25
party  DEMOCRAT  REPUBLICAN  Total    Majority
year                                          
1976        291         144    435    DEMOCRAT
1978        277         158    435    DEMOCRAT
1980        245         190    435    DEMOCRAT
1982        270         165    435    DEMOCRAT
1984        254         181    435    DEMOCRAT
1986        259         176    435    DEMOCRAT
1988        260         175    435    DEMOCRAT
1990        267         167    434    DEMOCRAT
1992        258         176    434    DEMOCRAT
1994        205         229    434  REPUBLICAN
1996        203         217    420  REPUBLICAN
1998        211         223    434  REPUBLICAN
2000        213         220    433  REPUBLICAN
2002        205         229    434  REPUBLICAN
2004        202         232    434  REPUBLICAN
2006        232         199    431    DEMOCRAT
2008        257         178    435    DEMOCRAT
2010        197         238    435  REPUBLICAN
2012        202         233    435  REPUBLICAN
2014      

In [103]:
# Incumbency advantage — incumbents consistently outperform challengers by a few points across all eras
# nationwide
shifted = seats_per_year["Majority"].shift(1)
majority_map = shifted.to_dict()
winners["nation_incumbent_party"] = winners["year"].map(majority_map)

# district
winners = winners.sort_values(['state_po', 'district', 'year'])
winners['district_incumbent_party'] = winners.groupby(['state_po', 'district'])['party'].shift(1)

winners['dem_won'] = (winners['party'] == 'DEMOCRAT').astype(int)
winners = winners[winners['district_incumbent_party'] != 'INDEPENDENT']

winners = pd.get_dummies(winners, columns=['nation_incumbent_party', 'district_incumbent_party'], drop_first=True)
print(winners.columns.tolist())

['year', 'state_po', 'district', 'party', 'candidatevotes', 'totalvotes', 'vote_share', 'dem_lean', 'dem_won', 'nation_incumbent_party_REPUBLICAN', 'district_incumbent_party_REPUBLICAN']


In [104]:
drop_cols = ['year', 'state_po', 'district', 'party', 'candidatevotes', 'totalvotes', 'vote_share', 'dem_won']

X = winners.drop(columns=drop_cols)
y = winners['dem_won']

print(X.shape)
print(X.columns.tolist())
print(X.isnull().sum())


(10831, 3)
['dem_lean', 'nation_incumbent_party_REPUBLICAN', 'district_incumbent_party_REPUBLICAN']
dem_lean                               1569
nation_incumbent_party_REPUBLICAN         0
district_incumbent_party_REPUBLICAN       0
dtype: int64


In [105]:
X = X.dropna()
y = y[X.index]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))
print(y_test.value_counts(normalize=True))


0.9093362115488397
1    0.556935
0    0.443065
Name: dem_won, dtype: float64


In [106]:
# who held each district going into 2026
incumbents_2026 = winners[winners['year'] == 2024][['state_po', 'district', 'party']].rename(columns={'party': 'district_incumbent_party'})

# dem lean for 2026 = average of 2022 and 2024
dem_lean_2026 = dems[dems['year'].isin([2022, 2024])].groupby(['state_po', 'district'])['dem_vote_share'].mean().reset_index().rename(columns={'dem_vote_share': 'dem_lean'})

pred_2026 = incumbents_2026.merge(dem_lean_2026, on=['state_po', 'district'], how='left')
pred_2026['nation_incumbent_party'] = 'REPUBLICAN'

pred_2026 = pd.get_dummies(pred_2026, columns=['nation_incumbent_party', 'district_incumbent_party'], drop_first=True)
pred_2026 = pred_2026.reindex(columns=X.columns, fill_value=0)

# fill missing dem_lean with median (districts where no Democrat ran in 2022 or 2024)
pred_2026['dem_lean'] = pred_2026['dem_lean'].fillna(pred_2026['dem_lean'].median())

predictions = model.predict(pred_2026)
print(f"Democrat seats: {predictions.sum()}")
print(f"Republican seats: {len(predictions) - predictions.sum()}")

for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature}: {coef:.3f}")


Democrat seats: 215
Republican seats: 219
dem_lean: 4.628
nation_incumbent_party_REPUBLICAN: 0.416
district_incumbent_party_REPUBLICAN: -3.586
